In [0]:
#!pip install "/Workspace/Users/claire_wilsonbarnes@next.co.uk/next-ads/wheels/dsutils-0.1.13-py3-none-any.whl"

In [0]:
import mlflow
import mlflow.spark

from pyspark.sql import functions as F
from pyspark.sql import Window

# Pipelining
from pyspark.ml.functions import vector_to_array

# Models
from dsutils.etl import (
    truncate_and_load,
    delete_from_and_load,
)

In [0]:
# Spark Performance Settings
spark.conf.set("spark.sql.shuffle.partitions", "auto")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

In [0]:
def get_widget_value(name, default):
    try:
        dbutils.widgets.text(name, str(default))
        value = dbutils.widgets.get(name)
        return value if value not in (None, "") else default
    except NameError:
        return default


def validate_positive_int(name, value):
    parsed_value = int(value)
    if parsed_value <= 0:
        raise ValueError(
            f"Invalid widget value for {name}: {value}. Value must be a positive integer."
        )
    return parsed_value

In [0]:
dbutils.widgets.text(
    name="catalog_schema_prefix",
    defaultValue="marketingdata_dev.claire_wilsonbarnes",
    label="catalog_schema_prefix",
)
dbutils.widgets.text(
    name="lookback_period", defaultValue="30", label="lookback_period"
)
dbutils.widgets.text(
    name="table_prefix",
    defaultValue="next_uk_nextAds_analytics_pctr",
    label="table_prefix",
)
dbutils.widgets.text(
    name="affinity_weighting_factor",
    defaultValue="4",
    label="affinity_weighting_factor",
)
dbutils.widgets.text(
    name="regressor_model_uri",
    defaultValue="models:/marketingdata_dev.ds_sandbox.nextads_analytics_pctr_affinity_regression_model/2",
    label="regressor_model_uri",
)
dbutils.widgets.text(
    name="classifier_model_uri",
    defaultValue="models:/marketingdata_dev.ds_sandbox.nextads_analytics_pctr_popularity_classification_model/2",
    label="classifier_model_uri",
)

In [0]:
catalog_schema_prefix = get_widget_value(
    "catalog_schema_prefix", "marketingdata_dev.claire_wilsonbarnes"
)
lookback_period = int(get_widget_value("lookback_period", "30"))
table_prefix = get_widget_value(
    "table_prefix", "next_uk_nextAds_analytics_pctr"
)
affinity_weighting_factor = int(
    get_widget_value("affinity_weighting_factor", "4")
)
regressor_model_uri = get_widget_value(
    "regressor_model_uri",
    "models:/marketingdata_dev.ds_sandbox.nextads_analytics_pctr_affinity_regression_model/2",
)
classifier_model_uri = get_widget_value(
    "classifier_model_uri",
    "models:/marketingdata_dev.ds_sandbox.nextads_analytics_pctr_popularity_classification_model/2",
)

FEATURE_TABLE = catalog_schema_prefix + "." + table_prefix + "_features"
TARGET_TABLE = catalog_schema_prefix + "." + table_prefix + "_predictions"
TARGET_TABLE_LATEST = (
    catalog_schema_prefix + "." + table_prefix + "_predictions_latest"
)

fill_zeros_columns = {
    "day_impressions": 0,
    "prior_day_impressions": 0,
    "week_impressions": 0,
    "prior_week_impressions": 0,
    "customer_total_clicks": 0,
    "customer_total_unique_adverts_clicked": 0,
    "customer_advert_previous_click_number": 0,
    "number_clicks_same_algodivision": 0,
    "view_highest_catid_weight": 0,
    "view_lift_adjusted": 0,
    "view_cs": 0,
    "purchase_highest_catid_weight": 0,
    "purchase_lift_adjusted": 0,
    "purchase_cs": 0,
}

popularity_smoothed_score_col = "popularity_smoothed_score"
regression_weighted_score_col = "regression_weighted_score"
popularity_click_prob_col = "popularity_prob_click"
popularity_probability_col = "probability"
regressor_predictions_col = "residual_predictions"
combined_score_col = "combined_weighted_score"
weighted_ranking_col = "weighted_ranking"
pk_cols = ["account_number", "UniqueAdID"]

target_cols = pk_cols + [
    popularity_smoothed_score_col,
    regression_weighted_score_col,
    popularity_click_prob_col,
    regressor_predictions_col,
    combined_score_col,
    weighted_ranking_col,
    "advert_impressions_30days",
    "advert_item_revenue",
    #'rundate',
]

# QA Thresholds
distribution_number_ads_threshold = 15
cumulative_coverage_threshold = 0.8
rank1_advert_coverage_threshold = 0.25

In [0]:
clicks_history_table = spark.table(
    catalog_schema_prefix + "." + table_prefix + "_training_clicks_lookback"
)
current_control_sheet = spark.table(
    "marketingdata_prod.warehouse.next_uk_nextads_control_sheet_latest"
)

ad_items_table = spark.table(
    "marketingdata_prod.warehouse.next_ads_sort_order_latest"
).select("uniqueAdID", "items")
baskets_table = spark.table(
    "marketingdata_prod.warehouse.baskets_uk_3y"
).filter(F.col("order_date") >= F.date_sub(F.current_date(), lookback_period))

In [0]:
# pctr_prediction_features
predictions_input = spark.table(FEATURE_TABLE)
predictions_input = predictions_input.fillna(fill_zeros_columns)

In [0]:
mlflow.set_registry_uri("databricks-uc")

try:
    popularity_model = mlflow.spark.load_model(classifier_model_uri)
    affinity_model = mlflow.spark.load_model(regressor_model_uri)

except Exception as e:
    print(f"Error in loading models :{e}")

In [0]:
popularity_scored_df = popularity_model.transform(predictions_input)
affinity_scored_df = affinity_model.transform(popularity_scored_df)

In [0]:
# Addition of advert click data over the last 30 days
dates_table = affinity_scored_df.select("rundate").distinct()

join_condition = clicks_history_table.date.between(
    F.date_sub(dates_table.rundate, lookback_period + 1),
    F.date_sub(dates_table.rundate, 1),
)

overall_ad_impressions = (
    clicks_history_table.join(dates_table, on=join_condition, how="inner")
    .groupBy("rundate", "title", "campaign", "versionnumber")
    .agg(
        F.sum("number_impressions").alias("num_impressions"),
        F.sum("number_clicks").alias("num_clicks"),
    )
)

overall_impressions = overall_ad_impressions.groupBy("rundate").agg(
    F.sum("num_impressions").alias("total_num_impressions"),
    F.sum("num_clicks").alias("total_num_clicks"),
    (F.sum("num_clicks") / F.sum("num_impressions")).alias(
        "global_clickthrough_rate"
    ),
    F.median("num_impressions").alias("median_impressions"),
)

In [0]:
join_condition_control_sheet = (
    (
        F.upper(current_control_sheet.CampaignNumber)
        == clicks_history_table.campaign
    )
    & (current_control_sheet.Title == clicks_history_table.title)
    & (
        clicks_history_table.versionnumber
        == F.regexp_extract(
            current_control_sheet.UniqueAdID, r"^.*_(V[1-9])_.*$", 1
        )
    )
)

global_ads_table = (
    overall_ad_impressions.select(
        "rundate", "title", "campaign", "versionnumber", "num_impressions"
    )
    .join(overall_impressions, on=["rundate"], how="inner")
    .join(current_control_sheet, on=join_condition_control_sheet)
    .withColumnsRenamed({"num_impressions": "advert_impressions_30days"})
    .select(
        overall_ad_impressions["rundate"],
        "uniqueAdID",
        "advert_impressions_30days",
        "median_impressions",
        "global_clickthrough_rate",
    )
    .distinct()
)

In [0]:
median_impressions=global_ads_table.filter(F.col("median_impressions").isNotNull()).dropDuplicates(["median_impressions"]).select("median_impressions").collect()[0][0]

In [0]:
## Addition of items from adverts revenue as a tiebreaker if necessary

ads_item_revenue_last_30days = (
    ad_items_table.join(
        baskets_table,
        how="left",
        on=[baskets_table["itemno"] == ad_items_table["items"]],
    )
    .groupBy("uniqueAdID")
    .agg(F.sum(F.col("s740orderstakenvalue")).alias("advert_item_revenue"))
)

In [0]:
combined_data=affinity_scored_df.join(
        global_ads_table, how="left", on=["rundate", "uniqueAdID"]
    ).join(ads_item_revenue_last_30days, how="left", on=["uniqueAdID"]).withColumn("advert_impressions_30days", F.coalesce(F.col("advert_impressions_30days"), F.lit(0))).withColumn("advert_item_revenue", F.coalesce(F.col("advert_item_revenue"),F.lit(0))).withColumn("median_impressions", F.lit(median_impressions))

predictions = (combined_data.withColumn(
        "popularity_scoring_multiplier",
        F.coalesce(
            (
                (F.col("advert_impressions_30days") + 1)
                / (
                    F.col("advert_impressions_30days")
                    + 1
                    + F.col("median_impressions")
                )
            ),
            F.lit(0),
        ),
    ).withColumn(
        popularity_click_prob_col,
        vector_to_array(F.col(popularity_probability_col))[1],
    ).withColumn(
        popularity_smoothed_score_col,
        F.col(popularity_click_prob_col)
        * F.col("popularity_scoring_multiplier"),
    ).withColumn(
        regression_weighted_score_col,
        F.col(regressor_predictions_col) * affinity_weighting_factor,
    ).withColumn(
        combined_score_col,
        F.col(regression_weighted_score_col)
        + F.col(popularity_smoothed_score_col),
    ).withColumn(
        weighted_ranking_col,
        F.dense_rank().over(
            Window.partitionBy("rundate", "account_number").orderBy(
                F.desc(combined_score_col),
                F.desc("advert_impressions_30days"),
                F.desc("advert_item_revenue"),
            )
        ),
    )
)

In [0]:
print("Loading output to table (latest)")
truncate_and_load(
    predictions.select(*target_cols),
    TARGET_TABLE_LATEST,
    pk_cols=pk_cols,
)
# predictions.select(*target_cols).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(TARGET_TABLE_LATEST)

In [0]:
print("Loading output to table")
delete_from_and_load(
    predictions.select(*target_cols),
    TARGET_TABLE,
    pk_cols=pk_cols,
    del_where={"rundate": "current_date()"},
)

In [0]:
prediction_scores = spark.table(TARGET_TABLE_LATEST)

In [0]:
## Run QA
errors = []

## Only 1 run date and is current date
distinct_rundates = prediction_scores.select("rundate").distinct().count()
try:
    assert distinct_rundates == 1, (
        f"Multiple rundates in {TARGET_TABLE_LATEST}"
    )
except AssertionError as e:
    errors.append(str(e))

# rank 1 & 2 have as many predictions as input customers
rank_1_number_predictions = prediction_scores.filter(
    F.col(weighted_ranking_col) == 1
).count()
rank_2_number_predictions = prediction_scores.filter(
    F.col(weighted_ranking_col) == 2
).count()
number_customers = (
    predictions_input.select("account_number").distinct().count()
)

try:
    assert rank_1_number_predictions == number_customers, (
        f"Number of rank 1 predictions {rank_1_number_predictions} does not match number of customers {number_customers}"
    )
except AssertionError as e:
    errors.append(str(e))

try:
    assert rank_2_number_predictions == number_customers, (
        f"Number of rank 2 predictions {rank_2_number_predictions} does not match number of customers {number_customers}"
    )
except AssertionError as e:
    errors.append(str(e))

# number of adverts covered in rank 1 & 2
filtered_df = prediction_scores.filter(F.col(weighted_ranking_col).isin(1, 2))
total_prediction_number = filtered_df.count()
total_number_adverts_rank12 = (
    filtered_df.select("uniqueAdID").distinct().count()
)
# At least 50% of adverts available represented in rank 1 & 2 positions
min_threshold_number_of_ads = round(
    predictions_input.select("uniqueAdID").distinct().count() / 2, 0
)

try:
    assert total_number_adverts_rank12 >= min_threshold_number_of_ads, (
        "Less than 50% of adverts available represented in rank 1 & 2 positions"
    )
except AssertionError as e:
    errors.append(str(e))

## Advert distribution
cumulative_sum_advert_coverage_window = Window.orderBy(
    F.desc("perc_total")
).rowsBetween(Window.unboundedPreceding, Window.currentRow)
aggregated_advert_rank1_distribution = (
    filtered_df.groupBy("uniqueAdID")
    .agg(
        F.count("uniqueAdID").alias("rank1_2"),
        (F.count("uniqueAdID") / total_prediction_number).alias("perc_total"),
    )
    .orderBy(F.desc("perc_total"))
    .withColumn(
        "cumulative_coverage",
        F.sum("perc_total").over(cumulative_sum_advert_coverage_window),
    )
)

# top 80% distribution
number_ads_cumulative_coverage = aggregated_advert_rank1_distribution.filter(
    F.col("cumulative_coverage") <= cumulative_coverage_threshold
).count()
try:
    assert (
        number_ads_cumulative_coverage >= distribution_number_ads_threshold
    ), (
        f"Less than {distribution_number_ads_threshold} ads cover {cumulative_coverage_threshold * 100}% of all rank 1 & 2 positions"
    )
except AssertionError as e:
    errors.append(str(e))


# max coverage percentage - does this meet the threshold

rank1_coverage_perc = aggregated_advert_rank1_distribution.select(
    "perc_total"
).collect()[0][0]
try:
    assert rank1_coverage_perc <= rank1_advert_coverage_threshold, (
        "Top ranked advert covers 25% or more of all rank 1 & 2 positions"
    )
except AssertionError as e:
    errors.append(str(e))

In [0]:
if errors:
    final_errors = "\n".join(errors)
    print(final_errors)
    raise AssertionError(final_errors)